Imports and Setup

We'll use pandas for data manipulation and seaborn for clean, professional-grade visuals.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set the visual style
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Helper functions for data cleaning
def convert_tokens(value):
    if pd.isna(value) or value == "": return 0
    value = str(value).upper().strip()
    if 'K' in value: return int(float(value.replace('K', '')) * 1_000)
    if 'M' in value: return int(float(value.replace('M', '')) * 1_000_000)
    try: return int(float(value))
    except: return 0

def convert_requests(value):
    val_str = str(value).lower().strip()
    if 'free' in val_str: return 1 
    try: return int(float(val_str))
    except: return 0

Data Loading & Preprocessing

This is the "heavy lifting" cell. It includes helper functions to convert 100.7K into 100700 and handle the free requests.

In [ ]:
# Load your CSV (Update the filename if necessary)
df = pd.read_csv('cursor_usage.csv')

# 1. Clean Dates and Numeric columns
df['Date'] = pd.to_datetime(df['Date'])
df['Tokens_Num'] = df['Total Tokens'].apply(convert_tokens)
df['Requests_Num'] = df['Requests'].apply(convert_requests)

# 2. Categorize Free vs Paid
df['Status'] = np.where(
    df['Requests'].astype(str).str.contains('free', case=False), 
    'Free', 'Paid'
)

# 3. Create Time Groupings
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df['Day'] = df['Date'].dt.date

print(f"Data Prepared: {len(df)} entries processed.")

Graphing Usage by Model

This cell creates a side-by-side comparison of Total Tokens and Total Requests for each model.

In [ ]:
model_usage = df.groupby('Model').agg({'Tokens_Num': 'sum', 'Requests_Num': 'sum'}).sort_values(by='Tokens_Num', ascending=False).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=model_usage, x='Tokens_Num', y='Model', ax=ax1, palette="viridis")
ax1.set_title('Total Tokens by Model', fontweight='bold')

sns.barplot(data=model_usage, x='Requests_Num', y='Model', ax=ax2, palette="magma")
ax2.set_title('Total Requests by Model', fontweight='bold')

plt.tight_layout()
plt.show()

Bonus - Timeline of Usage

Since you have a Date column, it’s often helpful to see when you are using these models most heavily.

In [ ]:
# Group by Date and Model
daily_usage = df.groupby([df['Date'].dt.date, 'Model'])['Tokens_Num'].sum().unstack().fillna(0)

daily_usage.plot(kind='area', stacked=True, figsize=(12, 6), alpha=0.7)
plt.title('Daily Token Usage by Model Over Time', fontsize=14, fontweight='bold')
plt.ylabel('Tokens')
plt.xlabel('Date')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Free vs Paid Usage Over Time

Add this new cell to generate a comparison graph. This uses a Stacked Area Chart, which is perfect for showing how the "mix" of your usage changes as the month progresses.

In [ ]:
# Group by Date and Status
status_over_time = df.groupby([df['Date'].dt.date, 'Status'])['Requests_Num'].sum().unstack().fillna(0)

# Ensure both columns exist for plotting even if one is empty
for col in ['Free', 'Paid']:
    if col not in status_over_time.columns:
        status_over_time[col] = 0

# Create the plot
plt.figure(figsize=(12, 6))
plt.stackplot(status_over_time.index, 
              status_over_time['Paid'], 
              status_over_time['Free'], 
              labels=['Paid (Included)', 'Free Tier'],
              colors=['#4C72B0', '#C44E52'], 
              alpha=0.8)

plt.title('Requests: Paid vs. Free Tier Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Number of Requests')
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

Total Tokens and Requests per Month

This cell calculates the sum of tokens for every month found in your CSV and displays them in a bar chart with readable labels (converting large numbers into K or M).

In [ ]:
# Aggregate Monthly Data
monthly_stats = df.groupby('Month').agg({'Tokens_Num': 'sum', 'Requests_Num': 'sum'}).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot Tokens per Month
sns.barplot(data=monthly_stats, x='Month', y='Tokens_Num', ax=ax1, palette='Blues_d')
ax1.set_title('Total Tokens per Month', fontweight='bold')
for i, v in enumerate(monthly_stats['Tokens_Num']):
    label = f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.1f}K"
    ax1.text(i, v, label, ha='center', va='bottom', fontweight='bold')

# Plot Requests per Month
sns.barplot(data=monthly_stats, x='Month', y='Requests_Num', ax=ax2, palette='Greens_d')
ax2.set_title('Total Requests per Month', fontweight='bold')
for i, v in enumerate(monthly_stats['Requests_Num']):
    ax2.text(i, v, f"{int(v)}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()